# Real PD Meniscus Segmentation — All Runs Comparison

Compares all locally available model predictions against ground truth masks on 8 labeled real PD patients.

| Column | Folder | What it is |
|---|---|---|
| **Baseline** | `real_pd_predictions_baseline/` | Pretrained 5-class model, no fine-tuning |
| **run_001** | `real_pd_predictions/` | v1 fine-tuning, 55 DESS patients |
| **run_002** | `real_pd_predictions_v2/` | v1 fine-tuning, old RegGAN ~155pt fake PD |
| **run_003** | `real_pd_predictions_run003/` | v1 + early stopping, old RegGAN |
| **run_002_v2** | `results/real_pd_predictions_run002_v2/` | **v2** fine-tuning, old RegGAN (aug + weighted CE + cosine LR) |
| **run_005** | `real_pd_predictions_run005/` | v1 fine-tuning, NEW RegGAN ep52 fake PD |
| **ep14-v2** | `real_pd_predictions_ep14_v2/` | **v2** fine-tuning, NEW RegGAN ep14 fake PD |

Ground truth labels: `1 = meniscus` (no lateral/medial split)  
Prediction labels: `1 = lateral, 2 = medial` → merged to `1 = meniscus` for Dice  
Baseline: 5-class model, class 4 = meniscus

In [1]:
import os
import glob
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import zoom

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
BASE = "/Users/anshikabajpai/Desktop/github/RegGAN_domain_adaptation_v2"

PD_DIR             = f"{BASE}/data/iu-dataset/pd-files"
GT_DIR             = f"{BASE}/data/iu-dataset/segmentation_masks"

PRED_BL_DIR        = f"{BASE}/real_pd_predictions_baseline"        # pretrained, no fine-tune
PRED_R001_DIR      = f"{BASE}/real_pd_predictions"                 # run_001: v1, 55pt DESS
PRED_R002_DIR      = f"{BASE}/real_pd_predictions_v2"              # run_002: v1, old RegGAN ~155pt
PRED_R003_DIR      = f"{BASE}/real_pd_predictions_run003"          # run_003: v1+ES, old RegGAN
PRED_R002V2_DIR    = f"{BASE}/results/real_pd_predictions_run002_v2"  # run_002_v2: v2, old RegGAN
PRED_R005_DIR      = f"{BASE}/real_pd_predictions_run005"          # run_005: v1, NEW RegGAN ep52
PRED_EP14_DIR      = f"{BASE}/real_pd_predictions_ep14_v2"         # ep14-v2: v2, NEW RegGAN ep14

In [3]:
def extract_patient_id(path):
    return Path(path).name.split("_")[0]

def patients_in_dir(directory, pattern="*.nii*"):
    files = glob.glob(os.path.join(directory, pattern))
    return {extract_patient_id(f): f for f in files
            if ".seg.nrrd" not in f and ".labels.csv" not in f}

def load_vol(path):
    img = nib.load(path)
    return np.asarray(img.get_fdata()), img.affine

# ── Mirror process_volume() from preprocess.py exactly ──────────────────────
def preprocess_gt_mask(path):
    """
    Apply the same RAS reorientation + in-plane isotropic resample + 384x384
    resize as process_volume() in preprocess.py — but with order=0 (nearest-
    neighbour) to preserve integer mask labels, and no intensity normalisation.

    process_volume steps:
      1. sitk.DICOMOrient → "RAS"
         sitk array is (S,A,R), transpose → (R,A,S)  ← slices on axis 0
      2. Resample A,S to isotropic (finer of sp_A, sp_S)
      3. Resize (A,S) → 384x384
      4. Normalise  ← SKIPPED for masks

    We replicate steps 1-3 using nibabel (available locally).
    nib.as_closest_canonical() is the nibabel equivalent of DICOMOrient RAS.
    nibabel get_fdata() returns (x,y,z) = (R,A,S) in RAS+ space, so axis 0
    is already the through-plane R axis — same result as the sitk transpose.
    """
    img = nib.load(path)
    img = nib.as_closest_canonical(img)          # step 1: reorient to RAS+
    arr = img.get_fdata().astype(np.float32)     # (R, A, S), slices on axis 0
    zooms = img.header.get_zooms()[:3]           # (sp_R, sp_A, sp_S)

    # step 2: resample A,S to isotropic (same condition as process_volume)
    sp_A, sp_S = float(zooms[1]), float(zooms[2])
    target_ip  = min(sp_A, sp_S)
    fa = sp_A / target_ip
    fs = sp_S / target_ip
    if abs(fa - 1.0) > 0.02 or abs(fs - 1.0) > 0.02:
        arr = zoom(arr, (1.0, fa, fs), order=0)

    # step 3: resize (A,S) to 384x384
    _, n_A, n_S = arr.shape
    if n_A != 384 or n_S != 384:
        arr = zoom(arr, (1.0, 384 / n_A, 384 / n_S), order=0)

    return arr   # (n_R, 384, 384), integer labels preserved

def preprocess_pd_image(path):
    """Same as process_volume() including intensity normalisation, for viz."""
    img = nib.load(path)
    img = nib.as_closest_canonical(img)
    arr = img.get_fdata().astype(np.float32)
    zooms = img.header.get_zooms()[:3]

    sp_A, sp_S = float(zooms[1]), float(zooms[2])
    target_ip  = min(sp_A, sp_S)
    fa, fs = sp_A / target_ip, sp_S / target_ip
    if abs(fa - 1.0) > 0.02 or abs(fs - 1.0) > 0.02:
        arr = zoom(arr, (1.0, fa, fs), order=3, prefilter=True)

    _, n_A, n_S = arr.shape
    if n_A != 384 or n_S != 384:
        arr = zoom(arr, (1.0, 384 / n_A, 384 / n_S), order=3)

    lo, hi = np.percentile(arr, 1), np.percentile(arr, 99)
    arr = np.clip(arr, lo, hi)
    arr = (arr - lo) / (hi - lo + 1e-8)
    return arr   # (n_R, 384, 384), normalised [0,1]

# ── Label merging ────────────────────────────────────────────────────────────
def merge_meniscus_v1v2(pred):
    """V1/V2: 3-class (0=bg, 1=lateral, 2=medial) → binary meniscus."""
    return (pred > 0).astype(bool)

def merge_meniscus_baseline(pred):
    """Baseline pitthexai: 5-class, class 4 = meniscus (both)."""
    return (pred == 4).astype(bool)

def dice_score(pred_bin, gt_bin):
    intersection = (pred_bin & gt_bin).sum()
    denom = pred_bin.sum() + gt_bin.sum()
    if denom == 0:
        return 1.0
    return float(2 * intersection / denom)

In [ ]:
# ── Find common patients across all runs ─────────────────────────────────────
gt_patients    = patients_in_dir(GT_DIR,          "*.nii")
bl_patients    = patients_in_dir(PRED_BL_DIR,     "*.nii.gz")
r001_patients  = patients_in_dir(PRED_R001_DIR,   "*.nii.gz")
r002_patients  = patients_in_dir(PRED_R002_DIR,   "*.nii.gz")
r003_patients  = patients_in_dir(PRED_R003_DIR,   "*.nii.gz")
r002v2_patients= patients_in_dir(PRED_R002V2_DIR, "*.nii.gz")
r005_patients  = patients_in_dir(PRED_R005_DIR,   "*.nii.gz")
ep14_patients  = patients_in_dir(PRED_EP14_DIR,   "*.nii.gz")

common = sorted(
    set(gt_patients)
    & set(bl_patients)
    & set(r001_patients)
    & set(r002_patients)
    & set(r003_patients)
    & set(r002v2_patients)
    & set(r005_patients)
    & set(ep14_patients)
)

for name, d in [("GT", gt_patients), ("Baseline", bl_patients),
                ("run_001", r001_patients), ("run_002", r002_patients),
                ("run_003", r003_patients), ("run_002_v2", r002v2_patients),
                ("run_005", r005_patients), ("ep14-v2", ep14_patients)]:
    print(f"{name:<15}: {len(d)} patients")

print(f"\nCommon across all: {len(common)}")
print(f"IDs: {common}")

In [ ]:
results = []

for pid in common:
    gt_path = gt_patients[pid]

    r001_raw,   _ = load_vol(r001_patients[pid])
    r002_raw,   _ = load_vol(r002_patients[pid])
    r003_raw,   _ = load_vol(r003_patients[pid])
    r002v2_raw, _ = load_vol(r002v2_patients[pid])
    r005_raw,   _ = load_vol(r005_patients[pid])
    ep14_raw,   _ = load_vol(ep14_patients[pid])
    bl_raw,     _ = load_vol(bl_patients[pid])

    gt = preprocess_gt_mask(gt_path)

    gt_bin     = (gt == 1).astype(bool)
    bl_bin     = merge_meniscus_baseline(bl_raw)
    r001_bin   = merge_meniscus_v1v2(r001_raw)
    r002_bin   = merge_meniscus_v1v2(r002_raw)
    r003_bin   = merge_meniscus_v1v2(r003_raw)
    r002v2_bin = merge_meniscus_v1v2(r002v2_raw)
    r005_bin   = merge_meniscus_v1v2(r005_raw)
    ep14_bin   = merge_meniscus_v1v2(ep14_raw)

    results.append({
        "patient":      pid,
        "dice_bl":      dice_score(bl_bin,     gt_bin),
        "dice_r001":    dice_score(r001_bin,   gt_bin),
        "dice_r002":    dice_score(r002_bin,   gt_bin),
        "dice_r003":    dice_score(r003_bin,   gt_bin),
        "dice_r002v2":  dice_score(r002v2_bin, gt_bin),
        "dice_r005":    dice_score(r005_bin,   gt_bin),
        "dice_ep14":    dice_score(ep14_bin,   gt_bin),
        "_gt": gt, "_bl": bl_raw,
        "_r001": r001_raw, "_r002": r002_raw, "_r003": r003_raw,
        "_r002v2": r002v2_raw, "_r005": r005_raw, "_ep14": ep14_raw,
    })
    r = results[-1]
    print(f"{pid}  bl={r['dice_bl']:.3f}  r001={r['dice_r001']:.3f}  "
          f"r002={r['dice_r002']:.3f}  r003={r['dice_r003']:.3f}  "
          f"r002v2={r['dice_r002v2']:.3f}  r005={r['dice_r005']:.3f}  ep14={r['dice_ep14']:.3f}")

mean_bl     = np.mean([r["dice_bl"]     for r in results])
mean_r001   = np.mean([r["dice_r001"]   for r in results])
mean_r002   = np.mean([r["dice_r002"]   for r in results])
mean_r003   = np.mean([r["dice_r003"]   for r in results])
mean_r002v2 = np.mean([r["dice_r002v2"] for r in results])
mean_r005   = np.mean([r["dice_r005"]   for r in results])
mean_ep14   = np.mean([r["dice_ep14"]   for r in results])

print(f"\nMEAN  bl={mean_bl:.4f}  r001={mean_r001:.4f}  r002={mean_r002:.4f}  "
      f"r003={mean_r003:.4f}  r002v2={mean_r002v2:.4f}  r005={mean_r005:.4f}  ep14={mean_ep14:.4f}")
print(f"\nBEST run: ", end="")
runs = {"Baseline": mean_bl, "run_001": mean_r001, "run_002": mean_r002,
        "run_003": mean_r003, "run_002_v2": mean_r002v2,
        "run_005": mean_r005, "ep14-v2": mean_ep14}
best_run = max(runs, key=runs.get)
print(f"{best_run}  Dice={runs[best_run]:.4f}")

In [ ]:
# ── Failure mode diagnosis — best run (run_002_v2) ───────────────────────────
print(f"{'Patient':<30} {'bl':>6} {'r001':>6} {'r002':>6} {'r003':>6} {'r002v2':>8} {'r005':>6} {'ep14':>6}  Failure mode (r002v2)")
print("-" * 120)

sorted_results = sorted(results, key=lambda x: x["dice_r002v2"])

for r in sorted_results:
    gt      = (r["_gt"] == 1).astype(bool)
    r002v2  = merge_meniscus_v1v2(r["_r002v2"])

    tp = (r002v2 & gt).sum()
    fp = (r002v2 & ~gt).sum()
    fn = (~r002v2 & gt).sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    pred_px   = int(r002v2.sum())

    if pred_px == 0:
        mode = "NO PREDICTION"
    elif recall < 0.4:
        mode = "UNDER-SEGMENTS"
    elif precision < 0.4:
        mode = "OVER-SEGMENTS"
    else:
        mode = "PARTIAL MATCH"

    print(f"{r['patient']:<30} {r['dice_bl']:>6.3f} {r['dice_r001']:>6.3f} "
          f"{r['dice_r002']:>6.3f} {r['dice_r003']:>6.3f} "
          f"{r['dice_r002v2']:>8.3f} {r['dice_r005']:>6.3f} {r['dice_ep14']:>6.3f}  {mode}")

print("-" * 120)
print(f"{'MEAN':<30} {mean_bl:>6.3f} {mean_r001:>6.3f} "
      f"{mean_r002:>6.3f} {mean_r003:>6.3f} "
      f"{mean_r002v2:>8.3f} {mean_r005:>6.3f} {mean_ep14:>6.3f}")

In [ ]:
# ── Per-slice Dice for 2 worst patients (run_002_v2) ─────────────────────────
worst2 = sorted(results, key=lambda x: x["dice_r002v2"])[:2]

fig, axes = plt.subplots(len(worst2), 1, figsize=(14, 4 * len(worst2)), facecolor="white")
if len(worst2) == 1:
    axes = [axes]

for ax, r in zip(axes, worst2):
    gt      = (r["_gt"] == 1).astype(bool)
    r002v2  = merge_meniscus_v1v2(r["_r002v2"])

    slice_dice = []
    for s in range(gt.shape[0]):
        tp = (r002v2[s] & gt[s]).sum()
        denom = r002v2[s].sum() + gt[s].sum()
        slice_dice.append(float(2 * tp / denom) if denom > 0 else float("nan"))

    slices = range(gt.shape[0])
    ax2 = ax.twinx()
    ax.bar(slices, gt.sum(axis=(1,2)),      alpha=0.3, color="cyan",    label="GT pixels")
    ax.bar(slices, r002v2.sum(axis=(1,2)),  alpha=0.3, color="magenta", label="run_002_v2 pred")
    ax2.plot(slices, slice_dice, color="red", linewidth=1.5, label="Slice Dice")
    ax2.axhline(r["dice_r002v2"], color="orange", linestyle="--",
                label=f"Mean Dice={r['dice_r002v2']:.3f}")
    ax2.set_ylim(0, 1.1); ax2.set_ylabel("Dice", color="red")
    ax.set_xlabel("Slice index"); ax.set_ylabel("Pixel count")
    ax.set_title(f"{r['patient']} — per-slice Dice (run_002_v2={r['dice_r002v2']:.3f})", fontweight="bold")
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{BASE}/per_slice_dice_worst.png", dpi=150)
plt.show()

In [ ]:
# ── Bar chart: Dice per patient — all runs ───────────────────────────────────
patients = [r["patient"] for r in results]
x = np.arange(len(patients))
w = 0.11   # 7 bars per group

RUNS = [
    ("Baseline",   [r["dice_bl"]     for r in results], mean_bl,     "tab:gray"),
    ("run_001\n55pt v1",  [r["dice_r001"]  for r in results], mean_r001,   "tab:orange"),
    ("run_002\noldReg v1",[r["dice_r002"]  for r in results], mean_r002,   "steelblue"),
    ("run_003\noldReg v1+ES",[r["dice_r003"] for r in results], mean_r003,  "cornflowerblue"),
    ("run_002_v2\noldReg v2",[r["dice_r002v2"] for r in results], mean_r002v2, "tab:green"),
    ("run_005\nnewReg v1",[r["dice_r005"]  for r in results], mean_r005,   "darkorange"),
    ("ep14-v2\nnewReg v2",[r["dice_ep14"]  for r in results], mean_ep14,   "tab:red"),
]

offsets = np.linspace(-3*w, 3*w, 7)

fig, ax = plt.subplots(figsize=(18, 6))
for i, (label, dice_vals, mean_val, color) in enumerate(RUNS):
    bars = ax.bar(x + offsets[i], dice_vals, w, label=f"{label} (μ={mean_val:.3f})", color=color, alpha=0.85)
    ax.axhline(mean_val, color=color, linestyle="--", alpha=0.5, linewidth=1)

ax.set_xticks(x)
ax.set_xticklabels(patients, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Dice Score")
ax.set_ylim(0, 1.05)
ax.set_title("Meniscus Dice vs Ground Truth — All Runs", fontweight="bold", fontsize=13)
ax.legend(loc="upper right", fontsize=7, ncol=2)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{BASE}/dice_comparison.png", dpi=150)
plt.show()

# Ranking
print("\nRanking by mean Dice:")
ranked = sorted([(label.replace('\n', ' '), mean_val) for label, _, mean_val, _ in RUNS], key=lambda x: -x[1])
for i, (name, score) in enumerate(ranked, 1):
    star = " ← BEST" if i == 1 else ""
    print(f"  {i}. {name:<30} {score:.4f}{star}")

In [ ]:
# ── Visual comparison — best 2 patients, key runs only ───────────────────────
# Show: PD image | GT | run_002 (v1) | run_002_v2 (v2) | run_005 (v1 newReg) | ep14-v2 (v2 newReg)
VIZ_PATIENTS = common[:2]

col_titles = ["PD Image", "Ground Truth",
              "run_002\noldReg v1", "run_002_v2\noldReg v2 ★",
              "run_005\nnewReg v1", "ep14-v2\nnewReg v2"]
cmaps_mask = ["Reds", "Blues", "Greens", "Oranges", "Purples"]

fig, axes = plt.subplots(len(VIZ_PATIENTS), 6, figsize=(24, 5 * len(VIZ_PATIENTS)))
if len(VIZ_PATIENTS) == 1:
    axes = axes.reshape(1, -1)

for row, pid in enumerate(VIZ_PATIENTS):
    r = next(x for x in results if x["patient"] == pid)
    gt       = r["_gt"]
    gt_bin   = (gt == 1).astype(bool)
    best_sl  = int(np.argmax(gt_bin.sum(axis=(1, 2))))

    masks_ordered = [
        gt_bin,
        merge_meniscus_v1v2(r["_r002"]),
        merge_meniscus_v1v2(r["_r002v2"]),
        merge_meniscus_v1v2(r["_r005"]),
        merge_meniscus_v1v2(r["_ep14"]),
    ]

    pd_files = glob.glob(os.path.join(PD_DIR, f"{pid}*.nii*"))
    if pd_files:
        pd_vol = preprocess_pd_image(pd_files[0])
        if pd_vol.shape[0] != gt.shape[0]:
            pd_vol = zoom(pd_vol, (gt.shape[0] / pd_vol.shape[0], 1.0, 1.0), order=3)
        pd_sl = pd_vol[best_sl]
    else:
        pd_sl = np.zeros((384, 384), dtype=float)

    for col in range(6):
        ax = axes[row, col]
        ax.imshow(pd_sl.T, cmap="gray", origin="lower", vmin=0, vmax=1)
        if col > 0:
            ax.imshow(np.ma.masked_equal(masks_ordered[col-1][best_sl].T.astype(float), 0),
                      cmap=cmaps_mask[col-1], origin="lower", alpha=0.6, vmin=0, vmax=1)
        if row == 0:
            ax.set_title(col_titles[col], fontsize=9, fontweight="bold")
        ax.set_ylabel(pid, fontsize=7)
        ax.axis("off")

plt.suptitle("Visual Comparison — Meniscus Segmentation on Real PD", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{BASE}/visual_comparison.png", dpi=150)
plt.show()

In [ ]:
# ── Boundary contour plot — GT vs run_002_v2 (best) ─────────────────────────
VIZ_PATIENTS = common[:2]

NEON = {"GT": "#00FFFF", "r002v2": "#39FF14", "ep14": "#FF00FF", "r005": "#FF6600"}
col_titles = ["PD Image", f"GT", "run_002_v2\n(oldReg v2)", "run_005\n(newReg v1)", "ep14-v2\n(newReg v2)", "GT+r002v2\noverlaid"]

fig, axes = plt.subplots(len(VIZ_PATIENTS), 6, figsize=(26, 5 * len(VIZ_PATIENTS)), facecolor="black")
if len(VIZ_PATIENTS) == 1:
    axes = axes.reshape(1, -1)

for row, pid in enumerate(VIZ_PATIENTS):
    r = next(x for x in results if x["patient"] == pid)
    gt       = r["_gt"]
    gt_bin   = (gt == 1).astype(bool)
    r002v2_b = merge_meniscus_v1v2(r["_r002v2"])
    r005_b   = merge_meniscus_v1v2(r["_r005"])
    ep14_b   = merge_meniscus_v1v2(r["_ep14"])
    best_sl  = int(np.argmax(gt_bin.sum(axis=(1, 2))))

    pd_files = glob.glob(os.path.join(PD_DIR, f"{pid}*.nii*"))
    pd_sl = np.zeros((384, 384), dtype=float)
    if pd_files:
        pd_vol = preprocess_pd_image(pd_files[0])
        if pd_vol.shape[0] != gt.shape[0]:
            pd_vol = zoom(pd_vol, (gt.shape[0] / pd_vol.shape[0], 1.0, 1.0), order=3)
        pd_sl = pd_vol[best_sl]

    masks_colors = [
        (gt_bin[best_sl],     NEON["GT"]),
        (r002v2_b[best_sl],   NEON["r002v2"]),
        (r005_b[best_sl],     NEON["r005"]),
        (ep14_b[best_sl],     NEON["ep14"]),
    ]

    for col in range(6):
        ax = axes[row, col]
        ax.set_facecolor("black")
        ax.imshow(pd_sl.T, cmap="gray", origin="lower", vmin=0, vmax=1)

        if col == 0:
            pass  # PD image only
        elif col <= 4:
            mask, color = masks_colors[col - 1]
            if mask.any():
                ax.contour(mask.T, levels=[0.5], colors=[color], linewidths=1.8)
        else:
            # last col: GT + r002v2 both overlaid
            if gt_bin[best_sl].any():
                ax.contour(gt_bin[best_sl].T,   levels=[0.5], colors=[NEON["GT"]],    linewidths=2.0, linestyles="solid")
            if r002v2_b[best_sl].any():
                ax.contour(r002v2_b[best_sl].T, levels=[0.5], colors=[NEON["r002v2"]], linewidths=2.0, linestyles="dashed")

        if row == 0:
            ax.set_title(col_titles[col], fontsize=9, fontweight="bold", color="white")
        ax.set_ylabel(pid, fontsize=7, color="white")
        ax.axis("off")

plt.suptitle("Boundary Contours — GT vs Predictions on Real PD", fontsize=13, fontweight="bold", color="white")
plt.tight_layout()
plt.savefig(f"{BASE}/boundary_comparison.png", dpi=150, facecolor="black")
plt.show()

In [ ]:
# ── GT vs run_002_v2 dual-boundary overlay — all 8 patients ──────────────────
GT_COLOR    = "#00FFFF"
R002V2_COLOR = "#39FF14"

COLS = 4
ROWS = int(np.ceil(len(common) / COLS))

fig, axes = plt.subplots(ROWS, COLS, figsize=(7 * COLS, 7 * ROWS), facecolor="black")
axes = np.array(axes).reshape(ROWS, COLS)

for idx, pid in enumerate(common):
    row, col = divmod(idx, COLS)
    ax = axes[row, col]

    r = next(x for x in results if x["patient"] == pid)
    gt       = r["_gt"]
    gt_bin   = (gt == 1).astype(bool)
    r002v2_b = merge_meniscus_v1v2(r["_r002v2"])
    best_sl  = int(np.argmax(gt_bin.sum(axis=(1, 2))))

    pd_files = glob.glob(os.path.join(PD_DIR, f"{pid}*.nii*"))
    pd_sl = np.zeros((384, 384), dtype=float)
    if pd_files:
        pd_vol = preprocess_pd_image(pd_files[0])
        if pd_vol.shape[0] != gt.shape[0]:
            pd_vol = zoom(pd_vol, (gt.shape[0] / pd_vol.shape[0], 1.0, 1.0), order=3)
        pd_sl = pd_vol[best_sl]

    ax.set_facecolor("black")
    ax.imshow(pd_sl.T, cmap="gray", origin="lower", vmin=0, vmax=1)
    if gt_bin[best_sl].any():
        ax.contour(gt_bin[best_sl].T,    levels=[0.5], colors=[GT_COLOR],     linewidths=2.5, linestyles="solid")
    if r002v2_b[best_sl].any():
        ax.contour(r002v2_b[best_sl].T,  levels=[0.5], colors=[R002V2_COLOR], linewidths=2.5, linestyles="dashed")

    ax.set_title(f"{pid}\nr002v2={r['dice_r002v2']:.3f}  ep14={r['dice_ep14']:.3f}",
                 fontsize=9, color="white", fontweight="bold")
    ax.axis("off")

for idx in range(len(common), ROWS * COLS):
    row, col = divmod(idx, COLS)
    axes[row, col].set_visible(False)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color=GT_COLOR,     linewidth=2.5, linestyle="solid",  label="Ground Truth"),
    Line2D([0], [0], color=R002V2_COLOR, linewidth=2.5, linestyle="dashed", label="run_002_v2 (oldReg v2)"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=2,
           fontsize=13, facecolor="black", labelcolor="white",
           framealpha=0.5, bbox_to_anchor=(0.5, 0.01))

plt.suptitle("GT vs run_002_v2 Boundary Overlay — All 8 Patients", fontsize=16, fontweight="bold", color="white")
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(f"{BASE}/gt_vs_v2_boundaries.png", dpi=150, facecolor="black", bbox_inches="tight")
plt.show()

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
print(f"{'Patient':<22} {'Baseline':>9} {'run_001':>9} {'run_002':>9} {'run_003':>9} {'run_002_v2':>11} {'run_005':>9} {'ep14-v2':>9}")
print("-" * 100)
for r in results:
    print(f"{r['patient']:<22} "
          f"{r['dice_bl']:>9.4f} {r['dice_r001']:>9.4f} {r['dice_r002']:>9.4f} "
          f"{r['dice_r003']:>9.4f} {r['dice_r002v2']:>11.4f} "
          f"{r['dice_r005']:>9.4f} {r['dice_ep14']:>9.4f}")
print("-" * 100)
print(f"{'MEAN':<22} "
      f"{mean_bl:>9.4f} {mean_r001:>9.4f} {mean_r002:>9.4f} "
      f"{mean_r003:>9.4f} {mean_r002v2:>11.4f} "
      f"{mean_r005:>9.4f} {mean_ep14:>9.4f}")

print()
print("── Improvements over baseline ──")
print(f"  run_001   vs baseline : +{(mean_r001-mean_bl)*100:.1f}%")
print(f"  run_002   vs baseline : +{(mean_r002-mean_bl)*100:.1f}%")
print(f"  run_003   vs baseline : +{(mean_r003-mean_bl)*100:.1f}%")
print(f"  run_002_v2 vs run_002 : +{(mean_r002v2-mean_r002)*100:.1f}%  (v1→v2 upgrade)")
print(f"  run_005   vs run_002  : +{(mean_r005-mean_r002)*100:.1f}%  (old→new RegGAN, v1)")
print(f"  ep14-v2   vs run_002  : +{(mean_ep14-mean_r002)*100:.1f}%  (old→new RegGAN, v2)")
print()

print("── Current ranking ──")
ranking = sorted([
    ("Baseline",   mean_bl),
    ("run_001",    mean_r001),
    ("run_002",    mean_r002),
    ("run_003",    mean_r003),
    ("run_002_v2", mean_r002v2),
    ("run_005",    mean_r005),
    ("ep14-v2",    mean_ep14),
], key=lambda x: -x[1])
for i, (name, score) in enumerate(ranking, 1):
    star = " ◄ BEST" if i == 1 else ""
    print(f"  {i}. {name:<15} {score:.4f}{star}")

print()
print("── Notes ──")
print("  v1  = CE+Dice, fixed LR=1e-5, no augmentation")
print("  v2  = weighted CE (bg=0.1, men=1.5) + augmentation + cosine LR (1e-5→1e-7)")
print("  oldReg = old RegGAN (~69 DESS volumes)")
print("  newReg = run_006 (155 DESS volumes, new training)")